In [2]:
import pandas as pd

In [3]:
from pathlib import Path
import pandas as pd

# =========================
# Global data loading (single source of truth)
# =========================
DATA_DIR = Path("data")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH  = DATA_DIR / "test.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print("train:", df_train.shape, "test:", df_test.shape)
df_train.head()

In [ ]:
df = df_train.copy()

In [5]:
import pandas as pd

def handle_missing_values(df):
    """
    Handle missing values for the House Prices dataset
    according to domain knowledge and Kaggle best practices.
    """
    df = df.copy()

    # =========================
    # 1. 缺失 = 没有该设施（类别）
    # =========================
    none_cols = [
        "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
        "GarageType", "GarageFinish", "GarageQual", "GarageCond",
        "BsmtExposure", "BsmtFinType2", "BsmtQual", "BsmtCond", "BsmtFinType1",
        "MasVnrType"
    ]
    for col in none_cols:
        df[col] = df[col].fillna("None")

    # =========================
    # 2. 缺失 = 没有该设施（数值）
    # =========================
    df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

    # =========================
    # 3. LotFrontage：按 Neighborhood 中位数填充
    # =========================
    df["LotFrontage"] = (
        df.groupby("Neighborhood")["LotFrontage"]
          .transform(lambda x: x.fillna(x.median()))
    )

    # =========================
    # 4. GarageYrBlt：没有车库 → 用 YearBuilt
    # =========================
    df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["YearBuilt"])

    # =========================
    # 5. Electrical：极少缺失 → 众数
    # =========================
    df["Electrical"] = df["Electrical"].fillna(
        df["Electrical"].mode()[0]
    )

    return df

In [6]:
import numpy as np
import pandas as pd

def create_features_v2(df: pd.DataFrame) -> pd.DataFrame:
    """
    House Prices feature engineering v2 (LightGBM / XGBoost friendly).

    Design goals:
    - Keep original variables (tree models can choose splits)
    - Add high-signal engineered features suggested by EDA:
      * selective log1p features (added as *_log, not replacing originals)
      * threshold/bucket features (cheap vs expensive style)
      * neighborhood × quality interaction (categorical)
      * amenity flags and composite scores
      * area totals and quality-weighted areas
      * lightweight cluster id (optional, off by default for safety)
    """
    df = df.copy()

    # -------------------------
    # 1) Time / age features
    # -------------------------
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["IsRemodeled"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)

    # Optional: simple "new house" indicator (threshold-ish)
    df["IsNewHouse"] = (df["HouseAge"] <= 5).astype(int)

    # -------------------------
    # 2) Area totals (strong signal)
    # -------------------------
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]

    df["TotalBathrooms"] = (
        df["FullBath"]
        + 0.5 * df["HalfBath"]
        + df["BsmtFullBath"]
        + 0.5 * df["BsmtHalfBath"]
    )

    # Outdoor space
    df["TotalPorchSF"] = (
        df["OpenPorchSF"]
        + df["EnclosedPorch"]
        + df["3SsnPorch"]
        + df["ScreenPorch"]
    )

    # -------------------------
    # 3) Amenity flags (missing-as-none already handled upstream)
    # -------------------------
    df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
    df["HasPool"] = (df["PoolArea"] > 0).astype(int)
    df["HasPorch"] = (df["TotalPorchSF"] > 0).astype(int)
    df["HasDeck"] = (df["WoodDeckSF"] > 0).astype(int)
    df["HasMasonryVeneer"] = (df["MasVnrArea"] > 0).astype(int)

    # Composite amenity score (cheap vs expensive segmentation hint)
    df["LuxuryAmenityScore"] = (
        df["HasPool"] + df["HasMasonryVeneer"] + df["HasDeck"] + df["HasPorch"] + df["HasFireplace"]
    )

    # -------------------------
    # 4) Quality × size interactions (tree models love these)
    # -------------------------
    df["QualGrLiv"] = df["OverallQual"] * df["GrLivArea"]
    df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
    df["QualGarage"] = df["OverallQual"] * df["GarageArea"]

    # -------------------------
    # 5) Threshold / bucket features (from your cheap vs expensive insight)
    # -------------------------
    df["IsHighQuality"] = (df["OverallQual"] >= 7).astype(int)
    df["IsLargeHouse"] = (df["GrLivArea"] >= 2000).astype(int)
    df["IsLuxury"] = ((df["OverallQual"] >= 8) & (df["GrLivArea"] >= 2500)).astype(int)

    # Size buckets (small / mid / large)
    df["GrLivAreaBin"] = pd.cut(
        df["GrLivArea"],
        bins=[-np.inf, 1200, 2000, np.inf],
        labels=["small", "mid", "large"]
    ).astype(str)

    # -------------------------
    # 6) Selective log features (add, don't replace)
    # -------------------------
    # Only for long-tail "scale" variables that tend to be monotonic with SalePrice.
    log_cols = [
        "LotArea",
        "LotFrontage",
        "GrLivArea",
        "TotalBsmtSF",
        "MasVnrArea",
        "GarageArea",
        "1stFlrSF",
        "2ndFlrSF",
    ]
    for c in log_cols:
        if c in df.columns:
            df[c + "_log"] = np.log1p(df[c])

    # -------------------------
    # 7) Neighborhood × Quality interaction (categorical)
    # -------------------------
    # This is powerful for LightGBM (native categorical) and still usable for XGBoost after one-hot.
    df["Neighborhood_Qual"] = df["Neighborhood"].astype(str) + "_" + df["OverallQual"].astype(str)

    # -------------------------
    # 8) Optional: lightweight clustering feature (OFF by default)
    # -------------------------
    # If you want to try it later, set enable_cluster=True and pass a fitted transformer externally.
    # (Keeping it off avoids data leakage/fit-on-full-data mistakes in notebooks.)
    # -------------------------

    return df


In [7]:
def cast_categoricals(df):
    df = df.copy()
    obj_cols = df.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        df[c] = df[c].astype("category")
    return df

In [8]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2,
                                       random_state=42)

In [9]:
df_train_clean = handle_missing_values(df_train)
df_test_clean  = handle_missing_values(df_test)

df_train_feat = create_features_v2(df_train_clean)
df_test_feat  = create_features_v2(df_test_clean)

# 关键：把类别列 cast 成 category
df_train_feat = cast_categoricals(df_train_feat)
df_test_feat  = cast_categoricals(df_test_feat)

In [10]:
import numpy as np

y = np.log1p(df_train["SalePrice"])
X = df_train_feat.drop(columns=["SalePrice"])

In [11]:
cat_cols = X.select_dtypes(include=["category"]).columns
for c in cat_cols:
    df_test_feat[c] = df_test_feat[c].cat.set_categories(X[c].cat.categories)

X_test = df_test_feat[X.columns]

In [11]:
import lightgbm as lgb
train_data = lgb.Dataset(
    X,
    label=y,
    categorical_feature=cat_cols.tolist(),
    free_raw_data=False
)

In [12]:
import numpy as np
import lightgbm as lgb

base_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "verbose": -1
}

candidates = [
    # 0) baseline (对照组)
    {"name": "baseline", **base_params},

    # 1) 稳定型 1（强烈推荐）
    {"name": "leaf30_l2_0.3", **base_params,
     "min_data_in_leaf": 30,
     "lambda_l2": 0.3},

    # 2) 稳定型 2
    {"name": "leaf40_l2_0.5", **base_params,
     "min_data_in_leaf": 40,
     "lambda_l2": 0.5},

    # 3) 稍增强（注意：num_leaves 和 max_depth 成对）
    {"name": "leaves63_depth8_leaf30_l2_0.3", **base_params,
     "num_leaves": 63,
     "max_depth": 8,
     "min_data_in_leaf": 30,
     "lambda_l2": 0.3},

    # 4) 采样扰动（更鲁棒，有时对 LB 友好）
    {"name": "sample075_leaf30_l2_0.3", **base_params,
     "feature_fraction": 0.75,
     "bagging_fraction": 0.75,
     "bagging_freq": 1,
     "min_data_in_leaf": 30,
     "lambda_l2": 0.3},
]

results = []

for i, p in enumerate(candidates, 1):
    print(f"\n=== [{i}/{len(candidates)}] {p['name']} ===")

    cv = lgb.cv(
        p,
        train_data,
        num_boost_round=5000,
        nfold=5,
        stratified=False,
        shuffle=True,
        seed=42,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(200)
        ]
    )

    best_iter = len(cv["valid rmse-mean"])
    best_rmse = cv["valid rmse-mean"][-1]
    results.append((p["name"], best_rmse, best_iter))

    print(f"Best iteration: {best_iter}")
    print(f"CV RMSE: {best_rmse:.5f}")

# 按 RMSE 排序输出
results_sorted = sorted(results, key=lambda x: x[1])
print("\n===== Summary (sorted by CV RMSE) =====")
for name, rmse, it in results_sorted:
    print(f"{name:28s} | RMSE={rmse:.5f} | best_iter={it}")


=== [1/5] baseline ===
Training until validation scores don't improve for 100 rounds
[200]	valid's rmse: 0.127779 + 0.0140448
Early stopping, best iteration is:
[271]	valid's rmse: 0.127238 + 0.0131341
Best iteration: 271
CV RMSE: 0.12724

=== [2/5] leaf30_l2_0.3 ===
Training until validation scores don't improve for 100 rounds
[200]	valid's rmse: 0.12894 + 0.0140084
[400]	valid's rmse: 0.128595 + 0.0122605
Early stopping, best iteration is:
[498]	valid's rmse: 0.128303 + 0.0116636
Best iteration: 498
CV RMSE: 0.12830

=== [3/5] leaf40_l2_0.5 ===
Training until validation scores don't improve for 100 rounds
[200]	valid's rmse: 0.130095 + 0.0162229
Early stopping, best iteration is:
[175]	valid's rmse: 0.129983 + 0.0163618
Best iteration: 175
CV RMSE: 0.12998

=== [4/5] leaves63_depth8_leaf30_l2_0.3 ===
Training until validation scores don't improve for 100 rounds
[200]	valid's rmse: 0.129566 + 0.0141958
[400]	valid's rmse: 0.128976 + 0.0122002
Early stopping, best iteration is:
[375]	

In [13]:
import numpy as np
import pandas as pd
import lightgbm as lgb

# =========================
# 0) 读取数据
# =========================
df = df_train.copy()
df_submit = df_test.copy()

# =========================
# 1) 特征工程（v2）
# =========================
train_feat = create_features_v2(handle_missing_values(df))
test_feat  = create_features_v2(handle_missing_values(df_submit))

# =========================
# 2) 目标变量（log1p）
# =========================
y_full = np.log1p(train_feat["SalePrice"])
X_full = train_feat.drop(columns=["SalePrice"]).copy()
X_test = test_feat.copy()

# =========================
# 3) object -> category
# =========================
obj_cols = X_full.select_dtypes(include=["object"]).columns
for c in obj_cols:
    X_full[c] = X_full[c].astype("category")
    X_test[c] = X_test[c].astype("category")

# =========================
# 4) 对齐类别 & 列顺序
# =========================
cat_cols = X_full.select_dtypes(include=["category"]).columns
for c in cat_cols:
    X_test[c] = X_test[c].cat.set_categories(X_full[c].cat.categories)

X_test = X_test[X_full.columns]

# =========================
# 5) 使用「最优参数」
# =========================
params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,

    # ⭐ tuned params
    "min_data_in_leaf": 30,
    "lambda_l2": 0.3,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 1,

    "seed": 42,
    "verbose": -1
}

best_iter = 266  # 来自 CV 的最优迭代轮数

# =========================
# 6) 全量训练
# =========================
train_data_full = lgb.Dataset(
    X_full,
    label=y_full,
    categorical_feature=cat_cols.tolist(),
    free_raw_data=False
)

model_full = lgb.train(
    params,
    train_data_full,
    num_boost_round=best_iter
)

# =========================
# 7) 预测 + 反变换
# =========================
pred_test_log = model_full.predict(X_test)
pred_test = np.expm1(pred_test_log)

# =========================
# 8) 生成提交文件
# =========================
submission = pd.DataFrame({
    "Id": df_submit["Id"],
    "SalePrice": np.clip(pred_test, 0, None)
})

submission.to_csv("submission_lgb_v2_tuned.csv", index=False)
submission.head()

,Id,SalePrice
0,1461,127875.260791
1,1462,157476.382881
2,1463,179476.367359
3,1464,188290.845783
4,1465,184507.182721


In [14]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# 0) 读取数据（按你之前变量名）
df = df_train.copy()
df_submit = df_test.copy()

# 1) 预处理（train + test 都要）
train_feat = create_features_v2(handle_missing_values(df))
test_feat  = create_features_v2(handle_missing_values(df_submit))

# 2) 目标变量（log1p）
y = np.log1p(train_feat["SalePrice"])
X = train_feat.drop(columns=["SalePrice"]).copy()
X_test = test_feat.copy()

# 3) One-hot 编码：拼接一起做，保证列完全一致
X_all = pd.concat([X, X_test], axis=0)
X_all = pd.get_dummies(X_all, drop_first=False)

X_enc = X_all.iloc[:len(X), :].copy()
X_test_enc = X_all.iloc[len(X):, :].copy()

# ===== baseline params（你现在的）=====
base_params = dict(
    n_estimators=10000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    min_child_weight=1,
    gamma=0.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ===== 5 组轻量微调候选 =====
candidates = [
    ("baseline", {**base_params}),

    # 1) 稳定型：更保守叶子 + 更强L2（最推荐）
    ("mcw3_l2_1.5", {**base_params, "min_child_weight": 3, "reg_lambda": 1.5}),

    # 2) 稳定型加强：再加 gamma
    ("mcw3_l2_1.5_g0.05", {**base_params, "min_child_weight": 3, "reg_lambda": 1.5, "gamma": 0.05}),

    # 3) 更保守：min_child_weight 更大
    ("mcw5_l2_2.0_g0.05", {**base_params, "min_child_weight": 5, "reg_lambda": 2.0, "gamma": 0.05}),

    # 4) 采样扰动：更鲁棒（有时对 LB 有帮助）
    ("sample075_mcw3_l2_1.5", {**base_params,
                              "subsample": 0.75, "colsample_bytree": 0.75,
                              "min_child_weight": 3, "reg_lambda": 1.5}),
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

all_results = []

for name, params in candidates:
    rmse_list = []
    best_iters = []

    print(f"\n=== {name} ===")
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_enc), 1):
        X_tr, X_va = X_enc.iloc[tr_idx], X_enc.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = XGBRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
            early_stopping_rounds=200
        )

        pred_va = model.predict(X_va)
        rmse = np.sqrt(mean_squared_error(y_va, pred_va))
        rmse_list.append(rmse)
        best_iters.append(model.best_iteration)

        print(f"Fold {fold}: RMSE={rmse:.5f}, best_iter={model.best_iteration}")

    mean_rmse = float(np.mean(rmse_list))
    std_rmse  = float(np.std(rmse_list))
    best_iter_final = int(np.mean(best_iters))

    all_results.append((name, mean_rmse, std_rmse, best_iter_final))
    print(f"{name} | CV RMSE: {mean_rmse:.5f} ± {std_rmse:.5f} | avg_best_iter={best_iter_final}")

# 排序汇总
all_results = sorted(all_results, key=lambda x: x[1])
print("\n===== Summary (sorted by CV RMSE) =====")
for name, mean_rmse, std_rmse, it in all_results:
    print(f"{name:25s} | RMSE={mean_rmse:.5f} ± {std_rmse:.5f} | best_iter≈{it}")

# 6) 用全量数据训练最终 XGBoost（用平均 best_iter）
#model_full = XGBRegressor(**{**xgb_params, "n_estimators": best_iter_final})
#model_full.fit(X_enc, y)

# 7) 预测 test.csv 并生成提交
#pred_test_log = model_full.predict(X_test_enc)
#pred_test = np.expm1(pred_test_log)

#submission_xgb = pd.DataFrame({
    #"Id": df_submit["Id"],
    #"SalePrice": np.clip(pred_test, 0, None)
#})

#submission_xgb.to_csv("submission_xgb.csv", index=False)
#submission_xgb.head()



=== baseline ===


C:\Users\23517\anaconda3\envs\tf\lib\site-packages\xgboost\sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold 1: RMSE=0.13128, best_iter=415


C:\Users\23517\anaconda3\envs\tf\lib\site-packages\xgboost\sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold 2: RMSE=0.11821, best_iter=541


...

Fold 3: RMSE=0.15939, best_iter=908


C:\Users\23517\anaconda3\envs\tf\lib\site-packages\xgboost\sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold 4: RMSE=0.11854, best_iter=794


C:\Users\23517\anaconda3\envs\tf\lib\site-packages\xgboost\sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Fold 5: RMSE=0.10697, best_iter=578
sample075_mcw3_l2_1.5 | CV RMSE: 0.12643 ± 0.01824 | avg_best_iter=637

===== Summary (sorted by CV RMSE) =====
baseline                  | RMSE=0.12597 ± 0.01608 | best_iter≈1018
sample075_mcw3_l2_1.5     | RMSE=0.12643 ± 0.01824 | best_iter≈637
mcw3_l2_1.5               | RMSE=0.12651 ± 0.01749 | best_iter≈876
mcw3_l2_1.5_g0.05         | RMSE=0.12952 ± 0.01806 | best_iter≈804
mcw5_l2_2.0_g0.05         | RMSE=0.12998 ± 0.01761 | best_iter≈886


In [15]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from xgboost import XGBRegressor

# =========================
# 0) Load
# =========================
df = df_train.copy()


# =========================
# 1) FE v2 (train + test)
# =========================
train_feat = create_features_v2(handle_missing_values(df))
test_feat  = create_features_v2(handle_missing_values(df_test))

y_full = np.log1p(train_feat["SalePrice"])

X_train_raw = train_feat.drop(columns=["SalePrice"]).copy()
X_test_raw  = test_feat.copy()

# ============================================================
# 2) LightGBM branch (native categorical)
# ============================================================
X_lgb = cast_categoricals(X_train_raw)
X_test_lgb = cast_categoricals(X_test_raw)

cat_cols = X_lgb.select_dtypes(include=["category"]).columns.tolist()

# Align category levels in test to train
for c in cat_cols:
    X_test_lgb[c] = X_test_lgb[c].cat.set_categories(X_lgb[c].cat.categories)

# Align column order
X_test_lgb = X_test_lgb[X_lgb.columns]

# ---- tuned LightGBM params / best_iter (sample075_leaf30_l2_0.3) ----
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,

    # ⭐ tuned points
    "min_data_in_leaf": 30,
    "lambda_l2": 0.3,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 1,

    "seed": 42,
    "verbose": -1
}

best_iter_lgb = 266  # ✅ tuned CV best iteration

dtrain_full = lgb.Dataset(
    X_lgb,
    label=y_full,
    categorical_feature=cat_cols,
    free_raw_data=False
)

model_lgb = lgb.train(
    lgb_params,
    dtrain_full,
    num_boost_round=best_iter_lgb
)

pred_lgb_log = model_lgb.predict(X_test_lgb)
pred_lgb = np.expm1(pred_lgb_log)

# ============================================================
# 3) XGBoost branch (one-hot)
# ============================================================
X_all = pd.concat([X_train_raw, X_test_raw], axis=0, ignore_index=True)
X_all = pd.get_dummies(X_all, drop_first=False)

X_enc = X_all.iloc[:len(X_train_raw), :].copy()
X_test_enc = X_all.iloc[len(X_train_raw):, :].copy()

xgb_params = dict(
    n_estimators=1018,      # ✅ 用你之前的 avg best_iter
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    min_child_weight=1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_xgb = XGBRegressor(**xgb_params)
model_xgb.fit(X_enc, y_full)

pred_xgb_log = model_xgb.predict(X_test_enc)
pred_xgb = np.expm1(pred_xgb_log)

# =========================
# 4) Save submissions
# =========================
def save_submission(pred, filename):
    sub = pd.DataFrame({
        "Id": df_test["Id"],
        "SalePrice": np.clip(pred, 0, None)
    })
    sub.to_csv(filename, index=False)
    print("Saved:", filename)

# Single models (sanity check)
save_submission(pred_lgb, "submission_lgb.csv")
save_submission(pred_xgb, "submission_xgb.csv")

# Ensembles (try a few)
for w_lgb in [0.50, 0.55, 0.60, 0.65, 0.70]:
    w_xgb = 1.0 - w_lgb
    pred_ens = w_lgb * pred_lgb + w_xgb * pred_xgb
    save_submission(pred_ens, f"submission_ens_lgb{w_lgb:.2f}_xgb{w_xgb:.2f}.csv")

Saved: submission_lgb.csv
Saved: submission_xgb.csv
Saved: submission_ens_lgb0.50_xgb0.50.csv
Saved: submission_ens_lgb0.55_xgb0.45.csv
Saved: submission_ens_lgb0.60_xgb0.40.csv
Saved: submission_ens_lgb0.65_xgb0.35.csv
Saved: submission_ens_lgb0.70_xgb0.30.csv


In [13]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

# =========================
# 0) 准备数据（你已经有这些函数）
# =========================
df_train = df_train.copy()
df_test  = df_test.copy()

train_feat = create_features_v2(handle_missing_values(df_train))
test_feat  = create_features_v2(handle_missing_values(df_test))

y = np.log1p(train_feat["SalePrice"]).reset_index(drop=True)

X_train_raw = train_feat.drop(columns=["SalePrice"]).reset_index(drop=True)
X_test_raw  = test_feat.reset_index(drop=True)

# =========================
# 1) LightGBM features (category)
# =========================
X_lgb = cast_categoricals(X_train_raw)
X_test_lgb = cast_categoricals(X_test_raw)
cat_cols = X_lgb.select_dtypes(include=["category"]).columns.tolist()

for c in cat_cols:
    X_test_lgb[c] = X_test_lgb[c].cat.set_categories(X_lgb[c].cat.categories)
X_test_lgb = X_test_lgb[X_lgb.columns]

# =========================
# 2) XGBoost features (one-hot)
# =========================
X_all = pd.concat([X_train_raw, X_test_raw], axis=0, ignore_index=True)
X_all = pd.get_dummies(X_all, drop_first=False)

X_xgb = X_all.iloc[:len(X_train_raw), :].copy()
X_test_xgb = X_all.iloc[len(X_train_raw):, :].copy()

# =========================
# 3) Base model params（用你验证过的）
# =========================
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,

    # ⭐ tuned points
    "min_data_in_leaf": 30,
    "lambda_l2": 0.3,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 1,

    "seed": 42,
    "verbose": -1
}

best_iter_lgb = 266  # ✅ tuned CV best iteration

xgb_params = dict(
    n_estimators=1018,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    min_child_weight=1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# =========================
# 4) OOF Stacking
# =========================
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_lgb))
oof_xgb = np.zeros(len(X_xgb))

test_pred_lgb_folds = []
test_pred_xgb_folds = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_lgb), 1):
    print(f"\n===== Fold {fold}/{n_splits} =====")

    # ---- LightGBM ----
    dtr = lgb.Dataset(X_lgb.iloc[tr_idx], label=y.iloc[tr_idx],
                      categorical_feature=cat_cols, free_raw_data=False)
    dva = lgb.Dataset(X_lgb.iloc[va_idx], label=y.iloc[va_idx],
                      categorical_feature=cat_cols, free_raw_data=False)

    model_lgb = lgb.train(
        lgb_params,
        dtr,
        num_boost_round=5000,
        valid_sets=[dva],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    oof_lgb[va_idx] = model_lgb.predict(X_lgb.iloc[va_idx], num_iteration=model_lgb.best_iteration)
    test_pred_lgb_folds.append(model_lgb.predict(X_test_lgb, num_iteration=model_lgb.best_iteration))

    # ---- XGBoost ----
    model_xgb = XGBRegressor(**xgb_params)
    model_xgb.fit(
        X_xgb.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X_xgb.iloc[va_idx], y.iloc[va_idx])],
        verbose=False
    )

    oof_xgb[va_idx] = model_xgb.predict(X_xgb.iloc[va_idx])
    test_pred_xgb_folds.append(model_xgb.predict(X_test_xgb))

# =========================
# 5) Meta model (Ridge)
# =========================
meta_X = np.column_stack([oof_lgb, oof_xgb])
meta_model = Ridge(alpha=1.0, random_state=42)
meta_model.fit(meta_X, y)

# test base preds (avg over folds)
test_lgb = np.mean(np.column_stack(test_pred_lgb_folds), axis=1)
test_xgb = np.mean(np.column_stack(test_pred_xgb_folds), axis=1)
meta_test_X = np.column_stack([test_lgb, test_xgb])

pred_stack_log = meta_model.predict(meta_test_X)
pred_stack = np.expm1(pred_stack_log)

submission_stack = pd.DataFrame({"Id": df_test["Id"], "SalePrice": np.clip(pred_stack, 0, None)})
submission_stack.to_csv("submission_stacking_ridge.csv", index=False)
submission_stack.head()


===== Fold 1/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[376]	valid_0's rmse: 0.135496

===== Fold 2/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[116]	valid_0's rmse: 0.113077

===== Fold 3/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[72]	valid_0's rmse: 0.151118

===== Fold 4/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[210]	valid_0's rmse: 0.124517

===== Fold 5/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[321]	valid_0's rmse: 0.113606


,Id,SalePrice
0,1461,125721.736136
1,1462,158632.453097
2,1463,180926.893027
3,1464,189410.572551
4,1465,182660.529731


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

df_train = df_train.copy()

# v2 FE
train_feat = create_features_v2(handle_missing_values(df_train))

y = np.log1p(train_feat["SalePrice"])
X = train_feat.drop(columns=["SalePrice"]).copy()

# 手动 one-hot（给 sklearn / RF / Linear 用）
X_enc = pd.get_dummies(X, drop_first=False)

In [ ]:
def cv_rmse(model, X, y, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rmses = []
    for fold, (tr, va) in enumerate(kf.split(X), 1):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y.iloc[tr], y.iloc[va]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)
        rmse = np.sqrt(mean_squared_error(y_va, pred))
        rmses.append(rmse)
        print(f"Fold {fold}: RMSE={rmse:.5f}")
    print(f"CV RMSE: {np.mean(rmses):.5f} ± {np.std(rmses):.5f}")
    return np.mean(rmses), np.std(rmses)

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=10.0)
cv_rmse(ridge, X_enc, y)

In [ ]:
from sklearn.linear_model import Lasso, ElasticNet

lasso = Lasso(alpha=0.0005, max_iter=20000)
cv_rmse(lasso, X_enc, y)

enet = ElasticNet(alpha=0.0008, l1_ratio=0.2, max_iter=20000)
cv_rmse(enet, X_enc, y)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=600,
    random_state=42,
    n_jobs=-1
)

cv_rmse(rf, X_enc, y)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

etr = ExtraTreesRegressor(
    n_estimators=1000,
    random_state=42,
    n_jobs=-1
)

cv_rmse(etr, X_enc, y)